In [1]:
!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/mssmartypants/rice-type-classification")

Dataset URL: https://www.kaggle.com/datasets/mssmartypants/rice-type-classification


100%|██████████| 888k/888k [00:00<00:00, 91.2MB/s]

In [14]:
import torch 
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [15]:
data_df = pd.read_csv("/content/rice-type-classification/riceClassification.csv")
data_df.dropna(inplace = True)
data_df.drop(["id"], axis=1, inplace=True)
print("Output possibilities: ", data_df["Class"].unique())
largestConvexArea = data_df["ConvexArea"].max()
minimumConvexArea = data_df["ConvexArea"].min()
print(f"Convex area range {minimumConvexArea, largestConvexArea}")
print("Data Shape (rows, cols): ", data_df.shape)
data_df.head()

Output possibilities:  [1 0]
Convex area range (2579, 11008)
Data Shape (rows, cols):  (18185, 11)


,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [16]:
original_df = data_df.copy()

for column in data_df.columns:
  data_df[column] = data_df[column] / data_df[column].abs().max()

data_df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,0.444368,0.503404,0.775435,0.744658,0.424873,0.666610,0.741661,0.537029,0.844997,0.368316,1.0
1,0.281293,0.407681,0.622653,0.750489,0.273892,0.530370,0.804230,0.409661,0.919215,0.371471,1.0
2,0.298531,0.416421,0.630442,0.756341,0.284520,0.546380,0.856278,0.412994,0.959862,0.374747,1.0
3,0.300979,0.420463,0.629049,0.764024,0.286791,0.548616,0.883772,0.414262,0.961818,0.379222,1.0
4,0.361704,0.464626,0.682901,0.775033,0.345385,0.601418,0.867808,0.452954,0.966836,0.386007,1.0


In [17]:
X = np.array(data_df.iloc[:, :-1])
Y = np.array(data_df.iloc[:, -1])

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.3)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5)

In [18]:
class dataset(Dataset):
  def __init__(self, X, Y):
    self.X = torch.tensor(X, dtype=torch.float32).to(device)
    self.Y = torch.tensor(Y, dtype=torch.float32).to(device)
  
  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.Y[index]
    

In [34]:
training_data = dataset(X_train, y_train)
validation_data = dataset(X_val, y_val)
testing_data = dataset(X_test, y_test)

BATCH_SIZE = 32
EPOCHS = 16
HIDDEN_NEURONS =12
LR = 1e-3


train_dataloader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True)
validation_dataloader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=True)
testing_dataloader = DataLoader(testing_data, batch_size=BATCH_SIZE, shuffle=True)

In [35]:
class RiceModel(nn.Module):
  def __init__(self):
    super(RiceModel, self).__init__()
    self.input_layer = nn.Linear(X.shape[1], HIDDEN_NEURONS)
    self.linear = nn.Linear(HIDDEN_NEURONS, 1)
    self.sigmoid = nn.Sigmoid()
  
  def forward(self, x):
    x = self.input_layer(x)
    x = self.linear(x)
    x = self.sigmoid(x)
    return x

In [36]:
rice_model = RiceModel().to(device)
summary(rice_model, (X.shape[1],))

criterion = nn.BCELoss()
optimizer = Adam(rice_model.parameters(), lr=LR)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 12]             132
            Linear-2                    [-1, 1]              13
           Sigmoid-3                    [-1, 1]               0
Total params: 145
Trainable params: 145
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [37]:
total_loss_train_plot = []
total_loss_validation_plot = []
total_acc_train_plot = []
total_acc_validation_plot = []

for epoch in range(EPOCHS):
  total_acc_train = 0
  total_loss_train = 0
  total_acc_val = 0
  total_loss_val = 0

  for data in train_dataloader:
    
    inputs, labels = data
    prediction = rice_model(inputs).squeeze(1)
    batch_loss = criterion(prediction, labels)
    total_loss_train += batch_loss.item()
    acc = ((prediction).round() == labels).sum().item()
    total_acc_train += acc
    batch_loss.backward()
    optimizer.step()
    optimizer.zero_grad()
  
  with torch.no_grad():
    for data in validation_dataloader:
      inputs, labels = data
      prediction = rice_model(inputs).squeeze(1)
      batch_loss = criterion(prediction, labels)
      total_loss_val += batch_loss.item()
      acc = ((prediction).round() == labels).sum().item()
      total_acc_val += acc
  
  total_loss_train_plot.append(round(total_loss_train/1000, 4))
  total_loss_validation_plot.append(round(total_loss_val/1000, 4))
  total_acc_train_plot.append(round(total_acc_train/(training_data.__len__())*100,4))
  total_acc_validation_plot.append(round(total_acc_val/(validation_data.__len__())*100, 4))

  print(f'Epoch no. {epoch+1} Training Loss: {total_loss_train/1000:.4f} Train Accuracy')

Epoch no. 1 Training Loss: 0.2333 Train Accuracy
Epoch no. 2 Training Loss: 0.1139 Train Accuracy
Epoch no. 3 Training Loss: 0.0535 Train Accuracy
Epoch no. 4 Training Loss: 0.0346 Train Accuracy
Epoch no. 5 Training Loss: 0.0269 Train Accuracy
Epoch no. 6 Training Loss: 0.0231 Train Accuracy
Epoch no. 7 Training Loss: 0.0209 Train Accuracy
Epoch no. 8 Training Loss: 0.0196 Train Accuracy
Epoch no. 9 Training Loss: 0.0188 Train Accuracy
Epoch no. 10 Training Loss: 0.0180 Train Accuracy
Epoch no. 11 Training Loss: 0.0176 Train Accuracy
Epoch no. 12 Training Loss: 0.0173 Train Accuracy
Epoch no. 13 Training Loss: 0.0172 Train Accuracy
Epoch no. 14 Training Loss: 0.0169 Train Accuracy
Epoch no. 15 Training Loss: 0.0168 Train Accuracy
Epoch no. 16 Training Loss: 0.0168 Train Accuracy


In [38]:
with torch.no_grad():
  total_loss_test = 0
  total_acc_test = 0

  for data in testing_dataloader:
    inputs, labels = data
    prediction = rice_model(inputs).squeeze(1)

    batch_loss_test = criterion((prediction), labels)
    total_loss_test += batch_loss_test.item()
    acc = ((prediction).round()==labels).sum().item()
    total_acc_test += acc

print(f"Accuracy score is: {round((total_acc_test/X_test.shape[0])*100, 2)} %")

Accuracy score is: 98.83 %
